# Landmark-Based Sign Language MLP Training

Trains a lightweight MLP on **MediaPipe hand landmark coordinates**.

## Why landmarks instead of raw images?
- Each hand = 21 landmarks × 3 (x, y, z) = **63 features**
- Background, lighting, skin tone are completely irrelevant
- Works with as few as 100–200 samples per class
- Training takes seconds, not hours

## Pipeline
```
collect_data.py → landmark_data.csv → this notebook → sign_landmark_model.pkl
```

**Run `collect_data.py` first to generate `landmark_data.csv`.**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline

DATA_PATH  = os.path.join(os.path.dirname(os.getcwd()), 'modules', 'sign', 'landmark_data.csv')
# If running from modules/sign/ directly:
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'landmark_data.csv'

MODEL_OUT  = os.path.join(os.path.dirname(DATA_PATH), 'sign_landmark_model.pkl')
print('Data path :', DATA_PATH)
print('Model out :', MODEL_OUT)

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f'Total rows : {len(df)}')
print(f'Columns    : {df.shape[1]}')
print()
print('Samples per class:')
print(df['label'].value_counts().sort_index())

In [ ]:
# ── Visualise class balance ───────────────────────────────────────────────
plt.figure(figsize=(10, 4))
df['label'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Sample count per sign class')
plt.xlabel('Sign')
plt.ylabel('Samples')
plt.tight_layout()
plt.show()

In [ ]:
# ── Prepare features & labels ─────────────────────────────────────────────
feature_cols = [c for c in df.columns if c != 'label']
X = df[feature_cols].values.astype(np.float32)   # shape (N, 63)
y_raw = df['label'].values

le = LabelEncoder()
y  = le.fit_transform(y_raw)

print('Feature shape :', X.shape)
print('Classes       :', list(le.classes_))

In [ ]:
# ── Train / val / test split ──────────────────────────────────────────────
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}')

In [ ]:
# ── Build pipeline: StandardScaler + MLP ─────────────────────────────────
# StandardScaler normalises each of the 63 landmark features to zero mean,
# unit variance — critical for MLPs to converge well.

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(
        hidden_layer_sizes=(256, 128, 64),
        activation='relu',
        solver='adam',
        max_iter=500,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        verbose=True,
    ))
])

pipeline.fit(X_train, y_train)
print('Training complete!')

In [ ]:
# ── Training loss curve ───────────────────────────────────────────────────
mlp = pipeline.named_steps['mlp']
plt.figure(figsize=(9, 4))
plt.plot(mlp.loss_curve_, label='Train loss')
if mlp.validation_scores_ is not None:
    plt.plot([1 - s for s in mlp.validation_scores_], label='Val error', linestyle='--')
plt.title('MLP Training Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss / Error')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Evaluation ────────────────────────────────────────────────────────────
val_acc  = accuracy_score(y_val,  pipeline.predict(X_val))
test_acc = accuracy_score(y_test, pipeline.predict(X_test))

print(f'Val  accuracy : {val_acc*100:.2f}%')
print(f'Test accuracy : {test_acc*100:.2f}%')
print()
print(classification_report(y_test, pipeline.predict(X_test), target_names=le.classes_))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(y_test, pipeline.predict(X_test))
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Landmark MLP')
plt.tight_layout()
plt.show()

In [ ]:
# ── Save model + label encoder ────────────────────────────────────────────
save_obj = {'pipeline': pipeline, 'label_encoder': le}
with open(MODEL_OUT, 'wb') as f:
    pickle.dump(save_obj, f)

print(f'Model saved → {MODEL_OUT}')
print(f'Classes     : {list(le.classes_)}')